In [0]:

from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from collections import defaultdict
import re

base = Path("/Volumes/workspace/sanchari_bronze/raw_youtube")
zip_path = base / "incoming/sanchari_day3_runs.zip"

assert zip_path.is_file(), "ZIP not found in volume"

expected_files = {"video_daily.jsonl", "manifest.json"}
run_files = defaultdict(set)

with ZipFile(zip_path) as archive:
    members = [
        info for info in archive.infolist()
        if not info.is_dir()
    ]

    for member in members:
        parts = PurePosixPath(member.filename).parts

        assert len(parts) == 2, member.filename
        run_id, filename = parts

        assert re.fullmatch(
            r"\d{8}T\d{6}Z_[0-9a-f]{8}", run_id
        ), f"Unexpected run ID: {run_id}"

        assert filename in expected_files
        assert filename not in run_files[run_id]
        run_files[run_id].add(filename)

    assert all(
        files == expected_files
        for files in run_files.values()
    )

    assert len(run_files) == 19
    assert len(members) == 38
    assert sum(m.file_size for m in members) < 250_000_000
    assert archive.testzip() is None, "ZIP integrity check failed"

print("ZIP integrity: PASS")
print("Extraction runs:", len(run_files))
print("Files:", len(members))
print("First run:", min(run_files))
print("Last run:", max(run_files))


ZIP integrity: PASS
Extraction runs: 19
Files: 38
First run: 20261028T205359Z_78d30800
Last run: 20261029T155023Z_0186e275


In [0]:

import shutil

runs_path = base / "runs"
created = 0
skipped = 0

with ZipFile(zip_path) as archive:
    for member in archive.infolist():
        if member.is_dir():
            continue

        parts = PurePosixPath(member.filename).parts
        assert len(parts) == 2

        run_id, filename = parts
        assert run_id in run_files
        assert filename in run_files[run_id]

        destination = runs_path / run_id / filename
        destination.parent.mkdir(parents=True, exist_ok=True)

        if destination.exists():
            assert destination.stat().st_size == member.file_size
            skipped += 1
            continue

        with archive.open(member) as source:
            with destination.open("xb") as target:
                shutil.copyfileobj(source, target)
        created += 1

print("Files extracted:", created)
print("Already present:", skipped)
print("Destination:", runs_path)


Files extracted: 38
Already present: 0
Destination: /Volumes/workspace/sanchari_bronze/raw_youtube/runs


In [0]:

from pyspark.sql import functions as F

jsonl_path = (
    "/Volumes/workspace/sanchari_bronze/"
    "raw_youtube/runs/*/video_daily.jsonl"
)

raw_df = (
    spark.read.format("text")
    .load(jsonl_path)
    .select(
        F.col("value").alias("raw_json"),
        F.col("_metadata.file_path").alias("source_file_path"),
        F.col("_metadata.file_modification_time").alias(
            "source_file_modified_at"
        )
    )
    .withColumn(
        "run_id",
        F.regexp_extract(
            F.col("source_file_path"),
            r"/runs/([^/]+)/",
            1
        )
    )
)

print("Total JSONL records:", raw_df.count())
print(
    "Runs containing records:",
    raw_df.select("run_id").distinct().count()
)

display(
    raw_df.groupBy("run_id")
    .count()
    .orderBy("run_id")
)


Total JSONL records: 37327
Runs containing records: 18


run_id,count
20261028T205359Z_78d30800,6
20261028T210824Z_c1db3000,3116
20261028T211355Z_70589085,535
20261029T154159Z_8bd5ee2f,850
20261029T154209Z_a8ecb933,3971
20261029T154214Z_1efc219d,1717
20261029T154219Z_d9058017,2231
20261029T154226Z_6c0f0494,2591
20261029T154232Z_d6d92c7d,2736
20261029T154239Z_0386ac55,2821


In [0]:

parsed_df = (
    spark.read
    .option("mode", "PERMISSIVE")
    .json(jsonl_path)
)

print("YouTube Analytics schema:")
parsed_df.printSchema()

print("Parsed rows:", parsed_df.count())

print("Column names:")
print(parsed_df.columns)


YouTube Analytics schema:
root
 |-- channel_id: string (nullable = true)
 |-- extract_run_id: string (nullable = true)
 |-- ingested_at_utc: string (nullable = true)
 |-- metrics: struct (nullable = true)
 |    |-- averageViewDuration: long (nullable = true)
 |    |-- comments: long (nullable = true)
 |    |-- estimatedMinutesWatched: long (nullable = true)
 |    |-- likes: long (nullable = true)
 |    |-- shares: long (nullable = true)
 |    |-- views: long (nullable = true)
 |-- raw_row: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- reporting_date: string (nullable = true)
 |-- source_endpoint: string (nullable = true)
 |-- video_id: string (nullable = true)

Parsed rows: 37327
Column names:
['channel_id', 'extract_run_id', 'ingested_at_utc', 'metrics', 'raw_row', 'reporting_date', 'source_endpoint', 'video_id']


In [0]:

from pyspark.sql import functions as F

bronze_df = (
    raw_df
    .withColumn(
        "channel_id",
        F.get_json_object("raw_json", "$.channel_id")
    )
    .withColumn(
        "video_id",
        F.get_json_object("raw_json", "$.video_id")
    )
    .withColumn(
        "reporting_date",
        F.get_json_object("raw_json", "$.reporting_date")
    )
    .withColumn(
        "extract_run_id_in_json",
        F.get_json_object("raw_json", "$.extract_run_id")
    )
    .withColumn(
        "record_sha256",
        F.sha2(F.col("raw_json"), 256)
    )
    .withColumn(
        "bronze_ingested_at",
        F.current_timestamp()
    )
    .select(
        "raw_json",
        "source_file_path",
        "source_file_modified_at",
        "run_id",
        "channel_id",
        "video_id",
        "reporting_date",
        "extract_run_id_in_json",
        "record_sha256",
        "bronze_ingested_at"
    )
)

total = bronze_df.count()

invalid = bronze_df.filter(
    F.col("run_id").isNull()
    | (F.col("run_id") == "")
    | F.col("video_id").isNull()
    | (F.col("video_id") == "")
    | F.col("reporting_date").isNull()
    | F.col("extract_run_id_in_json").isNull()
    | (
        F.col("extract_run_id_in_json")
        != F.col("run_id")
    )
).count()

duplicate_keys = (
    bronze_df
    .groupBy("run_id", "video_id", "reporting_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Bronze input records:", total)
print("Invalid metadata/key records:", invalid)
print("Duplicate video-day keys within runs:", duplicate_keys)

assert total == parsed_df.count(), "Raw/parsed counts differ"
assert invalid == 0, "Invalid records found"
assert duplicate_keys == 0, "Duplicate keys found"

print("Pre-ingestion validation: PASS")


Bronze input records: 37327
Invalid metadata/key records: 0
Duplicate video-day keys within runs: 0
Pre-ingestion validation: PASS


In [0]:

table_name = (
    "workspace.sanchari_bronze."
    "youtube_video_daily_raw"
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_name} (
    raw_json STRING,
    source_file_path STRING,
    source_file_modified_at TIMESTAMP,
    run_id STRING,
    channel_id STRING,
    video_id STRING,
    reporting_date STRING,
    extract_run_id_in_json STRING,
    record_sha256 STRING,
    bronze_ingested_at TIMESTAMP
)
USING DELTA
""")

bronze_df.createOrReplaceTempView(
    "incoming_youtube_bronze"
)

spark.sql(f"""
MERGE INTO {table_name} AS target
USING incoming_youtube_bronze AS source
ON target.run_id = source.run_id
AND target.video_id = source.video_id
AND target.reporting_date = source.reporting_date
WHEN NOT MATCHED THEN INSERT *
""")

print("Bronze Delta table:", table_name)
print("Rows stored:", spark.table(table_name).count())


Bronze Delta table: workspace.sanchari_bronze.youtube_video_daily_raw
Rows stored: 37327


In [0]:

spark.sql("""
SELECT
    COUNT(*) AS total_bronze_records,
    COUNT(DISTINCT run_id) AS nonempty_runs,
    COUNT(DISTINCT video_id) AS distinct_videos,
    MIN(reporting_date) AS earliest_date,
    MAX(reporting_date) AS latest_date
FROM workspace.sanchari_bronze.youtube_video_daily_raw
""").show()


+--------------------+-------------+---------------+-------------+-----------+
|total_bronze_records|nonempty_runs|distinct_videos|earliest_date|latest_date|
+--------------------+-------------+---------------+-------------+-----------+
|               37327|           18|           1239|   2023-05-25| 2026-10-24|
+--------------------+-------------+---------------+-------------+-----------+



In [0]:

table_name = "workspace.sanchari_bronze.youtube_video_daily_raw"

before = spark.table(table_name).count()

bronze_df.createOrReplaceTempView("incoming_youtube_bronze")

spark.sql(f"""
MERGE INTO {table_name} AS target
USING incoming_youtube_bronze AS source
ON target.run_id = source.run_id
AND target.video_id = source.video_id
AND target.reporting_date = source.reporting_date
WHEN NOT MATCHED THEN INSERT *
""")

after = spark.table(table_name).count()

print("Before rerun:", before)
print("After rerun:", after)
print("New rows:", after - before)

assert before == after, "Duplicate ingestion detected!"
print("Idempotency test: PASS")


Before rerun: 37327
After rerun: 37327
New rows: 0
Idempotency test: PASS


In [0]:

from pyspark.sql import functions as F

manifest_path = (
    "/Volumes/workspace/sanchari_bronze/"
    "raw_youtube/runs/*/manifest.json"
)

manifest_df = (
    spark.read
    .option("multiLine", "true")
    .json(manifest_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("manifest_file_path")
    )
    .withColumn(
        "run_id_from_path",
        F.regexp_extract(
            "manifest_file_path",
            r"/runs/([^/]+)/manifest\.json$",
            1
        )
    )
)

print("Manifest records:", manifest_df.count())
print(
    "Distinct manifest runs:",
    manifest_df.select("run_id_from_path").distinct().count()
)

manifest_df.printSchema()

bronze_run_ids = spark.table(table_name).select(
    F.col("run_id").alias("run_id_from_path")
).distinct()

empty_runs = (
    manifest_df.select("run_id_from_path")
    .join(bronze_run_ids, "run_id_from_path", "left_anti")
)

print("Runs without Bronze records:")
display(empty_runs)


Manifest records: 19
Distinct manifest runs: 19
root
 |-- api_pages: long (nullable = true)
 |-- channel_id: string (nullable = true)
 |-- column_headers: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- columnType: string (nullable = true)
 |    |    |-- dataType: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- day2_missing_video_ids: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- extract_run_id: string (nullable = true)
 |-- extracted_at_utc: string (nullable = true)
 |-- historical_backfill: boolean (nullable = true)
 |-- max_reported_date: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- record_count: long (nullable = true)
 |-- report_end_date: string (nullable = true)
 |-- report_start_date: string (nullable = true)
 |-- report_timezone: string (nullable = true)
 |-- requested_video_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- video_ids_reque

run_id_from_path
20261029T154119Z_54677c0c


In [0]:

from pyspark.sql import functions as F

bronze_counts = (
    spark.table(table_name)
    .groupBy("run_id")
    .agg(F.count("*").alias("bronze_record_count"))
)

audit_check = (
    manifest_df
    .join(
        bronze_counts,
        manifest_df.run_id_from_path == bronze_counts.run_id,
        "left"
    )
    .withColumn(
        "bronze_record_count",
        F.coalesce(
            F.col("bronze_record_count"),
            F.lit(0).cast("long")
        )
    )
    .withColumn(
        "id_matches",
        F.col("extract_run_id") == F.col("run_id_from_path")
    )
    .withColumn(
        "count_matches",
        F.col("record_count") == F.col("bronze_record_count")
    )
)

display(
    audit_check.select(
        "run_id_from_path",
        "mode",
        "status",
        "record_count",
        "bronze_record_count",
        "count_matches"
    ).orderBy("run_id_from_path")
)

mismatches = audit_check.filter(
    ~F.coalesce(F.col("id_matches"), F.lit(False))
    | ~F.coalesce(F.col("count_matches"), F.lit(False))
).count()

print("Manifest mismatches:", mismatches)
assert mismatches == 0, "Manifest reconciliation failed"
print("Manifest reconciliation: PASS")


run_id_from_path,mode,status,record_count,bronze_record_count,count_matches
20261028T205359Z_78d30800,null,completed,6,6,true
20261028T210824Z_c1db3000,backfill,completed,3116,3116,true
20261028T211355Z_70589085,incremental,completed,535,535,true
20261029T154119Z_54677c0c,backfill,completed,0,0,true
20261029T154159Z_8bd5ee2f,backfill,completed,850,850,true
20261029T154209Z_a8ecb933,backfill,completed,3971,3971,true
20261029T154214Z_1efc219d,backfill,completed,1717,1717,true
20261029T154219Z_d9058017,backfill,completed,2231,2231,true
20261029T154226Z_6c0f0494,backfill,completed,2591,2591,true
20261029T154232Z_d6d92c7d,backfill,completed,2736,2736,true


Manifest mismatches: 0
Manifest reconciliation: PASS


In [0]:

print("Channel IDs recorded in manifests:")
display(
    manifest_df.groupBy("channel_id").count()
)

print("Bronze rows missing channel_id:")
print(
    spark.table(table_name)
    .filter(
        F.col("channel_id").isNull()
        | (F.trim(F.col("channel_id")) == "")
    )
    .count()
)


Channel IDs recorded in manifests:


channel_id,count
UCGpXEis-qat1YG04joiG_Vw,18
null,1


Bronze rows missing channel_id:
6


In [0]:

audit_table = (
    "workspace.sanchari_bronze."
    "youtube_extraction_run_audit"
)

payload_columns = [
    c for c in manifest_df.columns
    if c not in ("manifest_file_path", "run_id_from_path")
]

audit_source = audit_check.select(
    F.col("run_id_from_path").alias("run_id"),
    F.col("manifest_file_path"),
    F.col("channel_id"),
    F.col("mode"),
    F.col("status"),
    F.col("historical_backfill"),
    F.col("record_count").alias("manifest_record_count"),
    F.col("bronze_record_count"),
    F.col("report_start_date"),
    F.col("report_end_date"),
    F.col("extracted_at_utc"),
    F.to_json(
        F.struct(*[F.col(c) for c in payload_columns])
    ).alias("manifest_json"),
    F.current_timestamp().alias("audit_loaded_at")
)

assert audit_source.count() == 19

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {audit_table} (
    run_id STRING,
    manifest_file_path STRING,
    channel_id STRING,
    mode STRING,
    status STRING,
    historical_backfill BOOLEAN,
    manifest_record_count BIGINT,
    bronze_record_count BIGINT,
    report_start_date STRING,
    report_end_date STRING,
    extracted_at_utc STRING,
    manifest_json STRING,
    audit_loaded_at TIMESTAMP
)
USING DELTA
""")

audit_source.createOrReplaceTempView(
    "incoming_youtube_run_audit"
)

spark.sql(f"""
MERGE INTO {audit_table} AS target
USING incoming_youtube_run_audit AS source
ON target.run_id = source.run_id
WHEN NOT MATCHED THEN INSERT *
""")

audit_count = spark.table(audit_table).count()

print("Audit table:", audit_table)
print("Audit records:", audit_count)

assert audit_count == 19
print("Audit ingestion: PASS")


Audit table: workspace.sanchari_bronze.youtube_extraction_run_audit
Audit records: 19
Audit ingestion: PASS


In [0]:

from pyspark.sql import functions as F

missing_channel_df = (
    spark.table(table_name)
    .filter(
        F.col("channel_id").isNull()
        | (F.trim(F.col("channel_id")) == "")
    )
)

print("Records missing channel ID:", missing_channel_df.count())

display(
    missing_channel_df
    .groupBy("run_id")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("video_id").alias("unique_videos"),
        F.min("reporting_date").alias("earliest_date"),
        F.max("reporting_date").alias("latest_date")
    )
)

display(
    missing_channel_df.select(
        "run_id",
        "video_id",
        "reporting_date"
    ).orderBy("reporting_date")
)


Records missing channel ID: 6


run_id,record_count,unique_videos,earliest_date,latest_date
20261028T205359Z_78d30800,6,1,2026-10-15,2026-10-23


run_id,video_id,reporting_date
20261028T205359Z_78d30800,3_MPPElISLA,2026-10-15
20261028T205359Z_78d30800,3_MPPElISLA,2026-10-16
20261028T205359Z_78d30800,3_MPPElISLA,2026-10-17
20261028T205359Z_78d30800,3_MPPElISLA,2026-10-18
20261028T205359Z_78d30800,3_MPPElISLA,2026-10-19
20261028T205359Z_78d30800,3_MPPElISLA,2026-10-23
